# 9. Conclusiones y limitaciones

In [1]:
# Preparación: acceso a los resultados versionados y a las funciones que generan las
# tablas y textos del informe.
import sys
sys.path.insert(0, '../ModeloOriginal')
from src import informe as inf

## 9.1. Hallazgos

In [2]:
# Cada hallazgo cita las cifras de los archivos de resultados (veredicto.json,
# metricas_prueba.csv, techo.json, incertidumbre_prueba.csv, ablacion.csv,
# semillas.csv, condicion4.json y auditoria_fuga.json).
inf.texto('hallazgos')

1. **Los benchmarks no fallan por falta de ajuste: tratan un escalón como ruido.** El error fuera de pliegue del mejor benchmark de regresión se concentra en cinco niveles separados 0,05 (Figura 5.1), que son el efecto de la variable excluida por fuga de información. Con esa estructura se puede calcular qué obtendría el mejor predictor posible, el que conociera el índice real de cada cliente: en prueba, RMSE 0,05911 y AUC 0,8313. Es el límite teórico de este informe, y los mejores benchmarks ya estaban a 0,00007 y 0,0030 de él.
2. **El modelo original recupera esa estructura sin ver la variable que la causa.** El criterio BIC elige $K = 5$ clases; sus pesos (0,050, 0,098, 0,149, 0,301, 0,402) difieren de la distribución real de `active_products` en menos de 0,001; la separación media entre desplazamientos es 0,0500 (la real es 0,05), y una vez descontado el escalón el error del índice en validación es 0,00076, unas 78 veces menor que el RMSE.
3. **En regresión, el original tiene la mejor media externa de los diez modelos**: RMSE 0,05887 frente a 0,05897 de XGBoost, por delante en 15 de 15 pliegues. En prueba obtiene 0,05912 (puesto 1 de 10), a 0,000010 del límite teórico. 2 de las 4 pruebas registradas rechazan la igualdad con el mejor benchmark a favor del original tras la corrección de Holm, y la guía pide tres: **no se demuestra superioridad**.
4. **En clasificación, el original no tiene la mejor media externa**: AUC 0,8313, puesto 5 de 10; XGBoost obtiene 0,8333. En prueba obtiene 0,8311 (puesto 1 de 10), a 0,00024 del límite teórico. Ninguna de las 4 pruebas registradas rechaza la igualdad con el mejor benchmark a favor del original tras la corrección de Holm, y la guía pide tres: **no se demuestra superioridad**.
5. **Qué habría obtenido el mejor predictor posible.** En clasificación, el predictor óptimo (el que conoce el índice real de cada cliente, que ningún modelo puede aprender sin la variable excluida) no rechaza la igualdad con XGBoost en ninguna de las 4 pruebas. En regresión, el predictor óptimo sí reúne 4 rechazos frente a XGBoost (P1, P2, P3 y P4): con una estimación casi perfecta del índice la condición era alcanzable, y el original se queda en 2.
6. **La incertidumbre es donde el modelo cambia el resultado.** Con cobertura del 90 %, su conjunto de predicción mide 0,0062 y el intervalo conforme del mejor benchmark, 0,1905: unas 31 veces más corto, con una cobertura medida de 89,9 % (H3 se cumple). En lugar de un intervalo ancho, el modelo dice en cuál de los niveles puede estar el puntaje. Frente a un conjunto del mismo tipo construido con los residuos de XGBoost (0,0436), unas 7 veces (comparación exploratoria).
7. **Las probabilidades salen calibradas, pero con bordes abruptos.** El error de calibración esperado del original es 0,008 sin recalibrar (XGBoost: 0,216). Aun así, su pérdida logarítmica (0,3688) no mejora la del mejor benchmark recalibrado (0,3675): H4 no se cumple, por unos pocos clientes en el borde de una meseta (sección 7.9).
8. **Qué aporta cada componente con todas las filas** (ablación, sección 5.9; lo que empeora la media externa con cada cambio, con su p ajustado): retirar la mezcla ($K = 1$), 0,000107 de RMSE (p = 0,040); retirar la mezcla ($K = 1$), 0,00211 de AUC (p = 0,652); aprender de la etiqueta en lugar del puntaje, 0,00284 de AUC (p = 0,652); cambiar la salida de la mezcla por la conforme empírica, 0,00075 de AUC (p = 0,729). En prueba ningún aporte es significativo.
9. **La clase latente da eficiencia muestral, no precisión cuando sobran datos.** Con el 10 % de las filas (3.118), el modelo completo obtiene un RMSE de 0,05896; sin la mezcla, 0,05971, y XGBoost, 0,05972. Con todas las filas los tres quedan en 0,05888, 0,05902 y 0,05896: con la mezcla, el modelo consigue con 3.118 filas lo que XGBoost consigue con 31.182 (sección 7.5; análisis exploratorio, añadido después de ver los resultados).
10. **Fuera de COFINFAD no hay ventaja.** En los 4 conjuntos de datos adicionales, el original no supera al mejor benchmark en ninguno, ni en regresión ni en clasificación; en regresión es peor de forma significativa en Vivienda de California y en clasificación es peor de forma significativa en Seguros médicos y Vivienda de California (sección 7.1). En 4 de ellos la búsqueda de hiperparámetros eligió para la regresión la variante sin mezcla, incluidos los construidos con una variable omitida: la ventaja que el protocolo esperaba ahí no apareció.
11. **El reajuste del modelo final casi no depende de la semilla, y el modelo cuesta más.** Al reajustar con diez semillas (misma partición y mismos hiperparámetros; solo cambia la semilla del modelo), el RMSE en prueba del original es 0,059116 ± 0,000002 y el de XGBoost, 0,059171 ± 0,000005. Ajustar el original tarda 5,8 s y XGBoost, 1,7 s: está escrito en `numpy`, sin compilar.
12. **La corrección de la recencia no cambia ningún dato.** Calculada solo con los clientes de entrenamiento, la fecha de corte es la misma (2023-12-29) y cambian 0 celdas: los resultados de las entregas anteriores siguen siendo válidos (capítulo 2).


## 9.2. Veredicto sobre R1, R2 y R3

La {ref}`Tabla 9.1 <tab-r123>` resume el estado de las tres condiciones de la guía y la
{ref}`Tabla 9.2 <tab-veredicto-final>` detalla la tercera.

(tab-r123)=
**Tabla 9.1.** Veredicto sobre las condiciones R1, R2 y R3, con la evidencia de cada una.

| Condición | Veredicto | Evidencia |
|---|---|---|
| **R1. Novedad en la aplicación** | Se cumple, con el alcance declarado en la sección 4.5 | Búsqueda en tres bases con trece cadenas, hecha dos veces, búsqueda por citación en los dos sentidos y revisión de repositorios: no se encontró ningún estudio que aplique a la fuga de clientes un boosting con una mezcla latente de desplazamientos ajustado por EM, ni otro algoritmo de esa familia. La regla de aprendizaje pertenece a una familia ya publicada (EM con paso de boosting); lo propio de BDL es el modelo al que se aplica, su regla de salida y su implementación ({ref}`Tabla 4.3 <tab-cercanos>`). El mecanismo está formulado de manera general (Algoritmo 1), tiene cinco propiedades demostradas y responde a una limitación identificada con un experimento ({ref}`Figura 5.1 <fig-escalones>`) |
| **R2. Implementación algorítmica propia** | Se cumple | `ModeloOriginal/src/original_model.py`: discretización, histogramas, árboles, EM de la mezcla, salida de excedencia y salida conforme, programados con `numpy` y `scipy`. El módulo no importa ningún modelo (lo comprueba una prueba unitaria). Las pruebas unitarias del modelo ({ref}`Tabla 5.4 <tab-pruebas-unitarias>`) y las del marco de experimentos, de las pruebas de hipótesis y del informe se ejecutan en integración continua |
| **R3. Superioridad estadísticamente verificada** | Ver la {ref}`Tabla 9.2 <tab-veredicto-final>` | Capítulos 6 y 7 |

(tab-veredicto-final)=
**Tabla 9.2.** Veredicto sobre R3 por tarea, con las cuatro condiciones de la guía (la misma de la sección 6.8).

In [3]:
# Tabla 9.2. Lee veredicto.json y condicion4.json: el veredicto se calcula a partir
# de las cuatro condiciones.
inf.tabla('tabla_veredicto')

| Tarea | Mejor benchmark | 1. Media externa mejor | 2. Al menos tres pruebas rechazan | 3. Efecto no despreciable | 4. Conjuntos adicionales | Veredicto R3 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| RMSE (regresión) | XGBoost | Sí (0,000099) | No (2 de 4) | Sí (δ = 0,20; *d* = 0,21) | No (mejor en 0 de 4; peor de forma significativa en Vivienda de California) | **No superior** |
| AUC (clasificación) | XGBoost | No (−0,00200) | No (0 de 4) | No (δ = −0,31; *d* = −0,51) | No (mejor en 0 de 4; peor de forma significativa en Seguros médicos, Vivienda de California) | **No superior** |

En las condiciones 1 y 3, el número entre paréntesis es la ventaja del original (positiva: a su favor) y el tamaño del efecto con ese mismo signo. En la 2, las pruebas que rechazan a favor del original con p ajustado < 0,05.


In [4]:
# Discusión del veredicto y análisis de la causa (el mismo texto de la sección 6.8).
inf.texto('veredicto_texto')

**Regresión (H1).** El RMSE medio del modelo original en los 15 pliegues externos es 0,05887, frente a 0,05897 de XGBoost, el mejor benchmark: una ventaja de 0,000099 (0,17 %), con el original por delante en 15 de los 15 pliegues. La condición 1 se cumple. De las 4 pruebas, rechazan la igualdad con XGBoost a favor del original P1 (Wilcoxon; p ajustado < 0,001) y P2 (*t* corregida; p ajustado = 0,001); no rechazan P3 (permutación pareada; p ajustado = 0,092) y P4 (bootstrap pareado; p ajustado = 0,076). La condición 2 pide tres rechazos: no se cumple. El tamaño del efecto entre los valores por pliegue (δ de Cliff = 0,20 y *d* de Cohen = 0,21) supera los umbrales de 0,147 y 0,2: la condición 3 se cumple. En los 4 conjuntos adicionales el original es mejor en 0 de 4; peor de forma significativa en Vivienda de California: la condición 4 no se cumple. **Veredicto: no superior.**

En prueba, el original obtiene 0,05912 (puesto 1 de 10) y XGBoost, 0,05917; el límite teórico es 0,05911, así que el benchmark queda a 0,000068 del límite y el original, a 0,000010. **Causa.** La diferencia es sistemática, pero tan pequeña que, con la corrección por comparaciones múltiples, las pruebas no alcanzan a detectarla todas: el benchmark ya estaba casi en el límite del conjunto y no dejaba margen para una diferencia mayor.

**Clasificación (H2).** El AUC medio del modelo original en los 15 pliegues externos es 0,8313 y el de XGBoost, el mejor benchmark, 0,8333: el original queda en el puesto 5 de 10, a 0,00200 del primero, y va por delante en 4 de los 15 pliegues. La condición 1 no se cumple. Ninguna de las 4 pruebas rechaza la igualdad con XGBoost a favor del original: P1 (Wilcoxon; p ajustado = 0,128), P2 (*t* corregida; p ajustado = 1,000), P3 (DeLong; p ajustado = 0,765) y P4 (McNemar; p ajustado = 1,000). La condición 2 pide tres rechazos: no se cumple. El tamaño del efecto (δ de Cliff = −0,31 y *d* de Cohen = −0,51) va en contra del original: la condición 3 no se cumple. En los 4 conjuntos adicionales el original es mejor en 0 de 4; peor de forma significativa en Seguros médicos, Vivienda de California: la condición 4 no se cumple. **Veredicto: no superior.**

En prueba, el original obtiene 0,8311 (puesto 1 de 10) y XGBoost, 0,8283; el límite teórico es 0,8313, así que el benchmark queda a 0,00301 del límite y el original, a 0,00024. **Análisis de la causa.** El espacio de búsqueda incluía la variante que aprende solo de la etiqueta (`usar_puntaje = False`), y la búsqueda interna la eligió en 7 de los 15 pliegues externos. En esos pliegues el original obtuvo un AUC medio de 0,8295 y XGBoost, 0,8332 (diferencia: −0,0037); en los 8 en que usó el puntaje, 0,8329 frente a 0,8334 (diferencia: −0,0005; por delante en 4 de 8). La desventaja en la media viene sobre todo de los pliegues en que el modelo no usó el puntaje. El modelo final, reajustado con toda la partición de entrenamiento, sí usa el puntaje. En cualquier caso, estas diferencias son menores que la desviación entre pliegues del propio benchmark (0,0041): con estos datos, los primeros modelos de la tabla no se distinguen entre sí en esta tarea. La ablación de la sección 5.9 mide por separado la variante sin puntaje.

**Hipótesis secundarias (no cuentan para R3).** H3: con cobertura nominal del 90 %, el conjunto de predicción del modelo original mide 0,0062 (la unión de 4 tramos) y el intervalo conforme de XGBoost, 0,1905; la cobertura medida en prueba es 89,9 %, dentro del error de muestreo (±0,6 puntos). H3 se cumple. H4: la pérdida logarítmica del original en prueba es 0,3688 y la de XGBoost con sus probabilidades recalibradas, 0,3675 (mínimo teórico: 0,3652). H4 no se cumple; la sección 7.9 muestra de dónde sale la diferencia.

**En conjunto, R3 no se cumple en ninguna de las dos tareas**: con el protocolo registrado, no se puede afirmar que el modelo original supere a los benchmarks.


## 9.3. Limitaciones

1. **Los datos son sintéticos.** COFINFAD no describe clientes reales. El escalón que BDL aprovecha existe porque el
   generador del conjunto resta una cantidad fija por producto activo; en datos reales un efecto así sería menos
   limpio. La evaluación en cuatro conjuntos públicos (capítulo 7), dos de mediciones reales y dos simulados, mide el
   comportamiento del modelo fuera de COFINFAD, pero no convierte estos resultados en evidencia sobre una fintech real.
2. **El margen de mejora era mínimo.** Los mejores benchmarks ya estaban a milésimas del límite teórico del conjunto
   (sección 6.4). Ningún modelo puede ofrecer una mejora grande en la métrica principal; lo que BDL añade es
   estructura (los niveles del puntaje) e incertidumbre mejor descrita, no precisión puntual.
3. **Con todas las filas, la mezcla apenas mueve la métrica.** La ablación (sección 5.9) muestra que retirar la clase
   latente cuesta del orden de una diezmilésima de RMSE cuando el modelo dispone de toda la muestra. Su aporte medible es de eficiencia
   muestral: con la décima parte de las filas es lo que separa al modelo de los benchmarks (sección 7.5). Ese análisis
   se hizo después de ver los resultados y es exploratorio.
4. **Supuestos del modelo.** Desplazamiento aditivo, independiente de los predictores y con la misma dispersión en
   todas las clases. Si la clase latente dependiera de los predictores haría falta una compuerta; si el efecto no fuera
   aditivo, una regresión por clase. Cuando los supuestos no se cumplen, el criterio BIC deja $K = 1$ y el modelo es un
   boosting de mínimos cuadrados.
5. **El puntaje continuo tiene que existir.** El modo de excedencia del clasificador necesita que la etiqueta sea un
   umbral de una variable continua observada en entrenamiento. Sin ella, el modelo aprende de la etiqueta y pierde su
   ventaja.
6. **La clasificación es frágil.** El resultado del clasificador depende de que la mezcla identifique bien los niveles:
   empeora con árboles de profundidad mayor que 1, con menos clases de las que hay o con un calentamiento corto, y con
   ninguna de las configuraciones probadas supera al mejor de los benchmarks de boosting evaluado con la misma receta
   ({ref}`Tabla 7.5 <tab-sensibilidad>`). Tolera peor que los benchmarks los valores atípicos en el puntaje y el ruido
   en los predictores (sección 7.4). En la validación anidada, la búsqueda llegó a elegir la variante sin puntaje en
   algunos pliegues (sección 6.8).
7. **Probabilidades con bordes abruptos y con mesetas.** La dispersión robusta hace que la probabilidad salte de una
   meseta a otra en un intervalo muy estrecho del índice. Un cliente mal situado en ese borde recibe una probabilidad
   casi segura y equivocada, y la pérdida logarítmica lo castiga (sección 7.9). Y dentro de una meseta el modelo no
   distingue a los clientes: cuando el corte de una campaña cae en una de ellas, a quién se contacta lo decide el
   desempate, y las métricas que dependen del orden entre empatados (la captura, la precisión media) cambian con la
   convención ({ref}`Tabla 6.11 <tab-empates>`). Le ocurre también al predictor óptimo: en este conjunto esos clientes
   son indistinguibles sin la variable excluida.
8. **La cobertura de los conjuntos de predicción es marginal.** Se garantiza sobre el total de clientes, no dentro de
   cada grupo: el nivel menos probable del escalón es el que queda fuera (sección 7.9).
9. **La cola de `tx_count`.** En menos del 2 % de los clientes el índice depende además del número de transacciones;
   ahí el modelo confunde parte de ese efecto con un cambio de nivel (sección 7.9).
10. **Costo.** Ajustar BDL cuesta varias veces más que ajustar XGBoost o LightGBM, porque está escrito en `numpy` sin
    compilar (sección 7.3).
11. **El conjunto de prueba ya se había usado** para evaluar los benchmarks en el Entregable 3. No intervino en el
    diseño del modelo original, y las repeticiones 2 y 3 de la validación anidada y los conjuntos adicionales son
    evidencia nueva para todos los modelos.
12. **Los pliegues del diseño forman parte de la comparación.** Las cinco iteraciones de diseño se evaluaron en los
    cinco pliegues externos de la semilla 42, que son cinco de los quince de la comparación confirmatoria (sección 6.1).
    En ellos el resultado puede favorecer al original; las repeticiones 2 y 3 y la partición de prueba no intervinieron
    en el diseño.
13. **El registro previo solo está certificado desde su publicación.** La etiqueta del protocolo se creó antes de la
    primera corrida, pero se publicó horas después, cuando ya existían los resultados de la primera repetición y de la
    prueba (sección 6.2). Para esa parte, la anterioridad del protocolo descansa en la fecha local de la etiqueta.
14. **La evidencia fuera de COFINFAD es limitada.** Los conjuntos adicionales son cuatro, pequeños o medianos, y lo
    que se obtiene en ellos (sección 7.1) no respalda una ventaja general: el mecanismo solo puede aportar cuando el
    objetivo tiene de verdad un desplazamiento discreto y bien separado, y cuando no lo encuentra el modelo se reduce a
    un boosting de mínimos cuadrados.
15. **Los benchmarks recientes no son los más nuevos.** LightGBM es de 2017 y la EBM, en su implementación actual, de
    2019. No se compararon redes neuronales para datos tabulares ni modelos preentrenados, y la búsqueda de los
    benchmarks no parte de sus hiperparámetros recomendados, aunque se midieron aparte (sección 3.6).

## 9.4. Riesgos y usos no recomendados

- **No es una probabilidad de fuga observada.** «Riesgo alto» es el cuarto superior de un puntaje de propensión; el
  modelo reproduce ese puntaje, no el comportamiento real de los clientes.
- **No usar para decidir sobre personas sin validación externa.** Antes de usarlo en una fintech habría que reentrenar
  y validar en el tiempo con fuga observada, y revisar el trato a los distintos grupos (la edad es la variable de más
  peso).
- **No extrapolar.** Fuera del rango de los datos de entrenamiento el modelo asigna el primer o el último bin de cada
  variable.

## 9.5. Trabajo futuro

- Hacer que los pesos de las clases dependan de los predictores (una compuerta por boosting) y permitir una dispersión
  distinta por clase.
- Elegir la dispersión de la probabilidad de excedencia con una regla de puntuación propia (la pérdida logarítmica en
  validación), para suavizar los bordes de las mesetas sin perder el orden.
- Conjuntos de predicción con cobertura condicionada al nivel del escalón.
- Incorporar el costo de cada acción de retención para fijar el umbral y la capacidad de la campaña.
- Compilar el bucle de histogramas para reducir el tiempo de ajuste.
- Validar con datos reales y con fuga observada, en una partición temporal.

(sec-declaracion)=
## 9.6. Declaración de autoría y uso de herramientas

Los autores de este trabajo son Valerie Martínez Pulido, Abrahan Basto Martínez y Rubén Esguerra Fernández, estudiantes
del pregrado en Ciencia de Datos de la Universidad del Norte.

**Uso de herramientas de inteligencia artificial.** En este entregable se usó un asistente de programación basado en
inteligencia artificial generativa (Claude Code, de Anthropic) como apoyo en:

- el análisis de la retroalimentación del docente y de la guía del curso;
- el diseño del modelo original y los experimentos piloto que lo motivaron;
- la programación del modelo, del marco de experimentos, de las pruebas unitarias y del dashboard;
- la ejecución de los experimentos y la generación de tablas y figuras;
- la búsqueda bibliográfica reproducible y la redacción y revisión de este libro.

El código del modelo original no se copió de ningún repositorio ni librería: se escribió para este proyecto. El equipo
revisó el trabajo, lo ejecutó, verificó sus resultados y responde por cada línea entregada; durante la sustentación
puede explicar y modificar cualquier parte del algoritmo, como exige la guía.

*Firmas del equipo:*

- Valerie Martínez Pulido: ____________________
- Abrahan Basto Martínez: ____________________
- Rubén Esguerra Fernández: ____________________